# Hugging Face モデルを Colab の GPU で動かして OnlineIDE から使う

このノートブックは、Colab の無料/Proの GPU 上で Hugging Face の Causal LM をロードし、
簡易的な推論サーバー（FastAPI）として公開します。公開には `cloudflared` の
クイックトンネルを使うので、アカウント登録や事前設定は不要です。

## 使い方
1. メニューの **ランタイム → ランタイムのタイプを変更 → GPU** を選択してから、
   上から順に全セルを実行してください。
2. 最後のセルの出力に表示される `COLAB_ENDPOINT_URL` と `COLAB_API_KEY` を、
   OnlineIDE リポジトリの `.env` に追記してください。
   ```
   COLAB_ENDPOINT_URL=https://xxxxx.trycloudflare.com
   COLAB_API_KEY=xxxxxxxxxxxxxxxxxxxx
   ```
3. OnlineIDE のチャットUIでモデルを「Hugging Face モデル (Colab GPU)」に切り替えると、
   このColabのGPUで生成された応答が返ってきます。

## 注意事項
- `cloudflared` のクイックトンネルは **誰でもアクセスできる公開URL** です。
  推測困難なAPIキーで保護していますが、URLとキーは他人に共有しないでください。
- Colab の無料枠はアイドル状態や長時間の接続で切断されます。切断されるとURLが
  無効になるので、その場合はこのノートブックを再実行して新しいURLを`.env`に
  設定し直してください。
- デフォルトのモデルは `Qwen/Qwen2.5-3B-Instruct`（ライセンス上ゲート無し）です。
  下の設定セルで `MODEL_ID` を変更すれば、他のHFモデルにも切り替えられます。
  ゲート付きモデル（Llama, Gemmaなど）を使う場合は `HF_TOKEN` にHugging Faceの
  アクセストークンを設定し、事前にモデルページでライセンスに同意してください。


## 1. パッケージのインストール

In [ ]:
!pip install -q -U transformers accelerate sentencepiece fastapi "uvicorn[standard]" nest_asyncio

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPUが見つかりません。ランタイム → ランタイムのタイプを変更 → GPU を選択してください。")


## 2. 設定

In [ ]:
import secrets

# 使用するHugging Faceモデル。ゲート付きモデルを使う場合は HF_TOKEN も設定してください。
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN = ""  # 例: "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"（不要なら空文字のまま）

# このColabサーバーを保護するAPIキー。毎回ランダムに生成されます。
API_KEY = secrets.token_urlsafe(24)

PORT = 8000


## 3. モデルのロード

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

token = HF_TOKEN or None

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=token,
    torch_dtype=torch.float16,
    device_map="cuda",
)
model.eval()
print(f"Loaded {MODEL_ID} on {model.device}")


## 4. 推論サーバー（FastAPI）の定義

OnlineIDE 側（`server.js`）がそのまま中継できるよう、`/generate` はGemini連携と
同じ `data: {"text": "..."}` 形式のSSEで応答を返します。


In [ ]:
import json
import threading

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import StreamingResponse
from transformers import TextIteratorStreamer

app = FastAPI()


def build_prompt(messages):
    chat = [
        {"role": "assistant" if m.get("role") == "model" else "user", "content": m.get("text", "")}
        for m in messages
        if m.get("text")
    ]
    return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)


def check_auth(request: Request):
    auth = request.headers.get("authorization", "")
    if auth != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}


@app.post("/generate")
async def generate(request: Request):
    check_auth(request)
    body = await request.json()
    messages = body.get("messages", [])
    if not messages:
        raise HTTPException(status_code=400, detail="messages is required")

    prompt = build_prompt(messages)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generate_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    thread = threading.Thread(target=model.generate, kwargs=generate_kwargs)
    thread.start()

    def event_stream():
        try:
            for token_text in streamer:
                if token_text:
                    yield f"data: {json.dumps({'text': token_text})}\n\n"
            yield "data: [DONE]\n\n"
        except Exception as e:
            yield f"data: {json.dumps({'error': str(e)})}\n\n"

    return StreamingResponse(event_stream(), media_type="text/event-stream")


## 5. サーバー起動と公開URLの発行

実行すると、このセルはトンネルURLが確立するまでブロックされます。URLが表示されたら
次に進んで大丈夫です（サーバーとトンネルはバックグラウンドで動き続けます）。


In [ ]:
import re
import subprocess

import nest_asyncio
import uvicorn

nest_asyncio.apply()


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

endpoint_url = None
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        endpoint_url = match.group(0)
        break

if not endpoint_url:
    print("トンネルURLの検出に失敗しました。上のセルからやり直してください。")
else:
    print("=" * 70)
    print("OnlineIDE の .env に以下を追記してください:")
    print()
    print(f"COLAB_ENDPOINT_URL={endpoint_url}")
    print(f"COLAB_API_KEY={API_KEY}")
    print("=" * 70)
    print("このノートブックを開いたまま（実行中のまま）にしてください。")
    print("閉じる/切断するとURLが無効になります。")


## (任意) 動作確認

別セルで直接 `/generate` を叩いて、ストリーミング応答を確認できます。


In [ ]:
import requests

resp = requests.post(
    f"http://localhost:{PORT}/generate",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={"messages": [{"role": "user", "text": "自己紹介してください"}]},
    stream=True,
)
for line in resp.iter_lines(decode_unicode=True):
    if line:
        print(line)
